# Módulo 03 · Aula 3 — Estatística descritiva

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Gráficos mostram; números medem. Esta aula é sobre os números que resumem uma variável e
a relação entre duas.

Não é uma aula de estatística teórica — é sobre **quando cada medida engana**. Toda
medida resumo joga informação fora (é para isso que serve), e a competência aqui está em
saber o que foi jogado fora.

Ao final você vai saber:

- escolher entre **média, mediana e moda**;
- medir dispersão com **desvio padrão, IQR e coeficiente de variação**;
- ler **quartis e percentis**, e identificar assimetria;
- distinguir **outlier** de **erro**;
- calcular e interpretar **correlação** — e por que ela não é causalidade.

**Tempo estimado:** 60 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "03_Visualizacao_EDA"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")

acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
acoes = acoes.sort_values(["ticker", "data"])
acoes["retorno_diario"] = acoes.groupby("ticker")["fechamento_ajustado"].pct_change() * 100

clientes = pd.read_csv("../data/clientes_corretora.csv").drop_duplicates()
clientes["patrimonio_investido"] = pd.to_numeric(
    clientes["patrimonio_investido"].astype(str)
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False).str.strip(),
    errors="coerce",
)
clientes.loc[~clientes["idade"].between(18, 110), "idade"] = np.nan
clientes["idade"] = clientes["idade"].fillna(clientes["idade"].median())
clientes["perfil_investidor"] = clientes["perfil_investidor"].str.strip().str.title()

print("Dados prontos:", acoes.shape, clientes.shape)

## 1. Medidas de centro

Três formas de responder "qual é o valor típico?", e elas **não** são intercambiáveis.

| Medida | Definição | Ponto forte | Ponto fraco |
|---|---|---|---|
| **Média** | soma ÷ quantidade | usa todos os valores; base de quase toda a estatística | sensível a extremos |
| **Mediana** | o valor do meio | robusta a extremos | ignora a magnitude dos valores |
| **Moda** | o valor mais frequente | única que serve para categorias | pode não existir ou ser múltipla |

In [ ]:
patrimonio = clientes["patrimonio_investido"].dropna()

print(f"Média   : R$ {patrimonio.mean():>12,.2f}")
print(f"Mediana : R$ {patrimonio.median():>12,.2f}")
print(f"Moda (perfil): {clientes['perfil_investidor'].mode()[0]}")

A média é bem maior que a mediana. Essa distância **é** a informação: ela diz que a
distribuição é assimétrica à direita, puxada por poucos clientes de patrimônio muito
alto.

Em números: se metade dos clientes tem menos de R$ 84 mil, dizer que "o cliente médio
tem R$ 123 mil" descreve um cliente que quase não existe.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

sns.histplot(patrimonio, bins=40, ax=ax, color="#8fb8d8")
ax.axvline(patrimonio.mean(), color="#c0392b", linewidth=2,
           label=f"Média: R$ {patrimonio.mean():,.0f}")
ax.axvline(patrimonio.median(), color="#27ae60", linewidth=2,
           label=f"Mediana: R$ {patrimonio.median():,.0f}")

ax.set_title("Média × mediana em uma distribuição assimétrica")
ax.set_xlabel("Patrimônio investido (R$)")
ax.set_ylabel("Número de clientes")
ax.legend()

plt.show()

### Uma demonstração da fragilidade da média

In [ ]:
salarios = pd.Series([3_000, 3_500, 4_000, 4_200, 4_800, 5_000, 5_500])

print("Sala com 7 pessoas:")
print(f"  Média   : R$ {salarios.mean():,.2f}")
print(f"  Mediana : R$ {salarios.median():,.2f}")

com_chefe = pd.concat([salarios, pd.Series([250_000])], ignore_index=True)

print("\nO chefe entra na sala:")
print(f"  Média   : R$ {com_chefe.mean():,.2f}   <- subiu {com_chefe.mean()/salarios.mean()-1:.0%}")
print(f"  Mediana : R$ {com_chefe.median():,.2f}   <- praticamente igual")

> **A regra prática:** em distribuições assimétricas — renda, patrimônio, faturamento,
> tempo de resposta, número de seguidores — **reporte a mediana**. Reportar a média não
> é errado, mas é incompleto: informe as duas e a diferença entre elas conta a história.

## 2. Medidas de dispersão

Dois grupos podem ter a mesma média e serem completamente diferentes. Dispersão mede o
quanto os valores se espalham em torno do centro — e, em finanças, dispersão **é** a
definição operacional de risco.

In [ ]:
conservador = pd.Series([98, 99, 100, 101, 102])
arriscado = pd.Series([40, 70, 100, 130, 160])

print("Mesma média, dispersões opostas:")
for nome, serie in [("conservador", conservador), ("arriscado", arriscado)]:
    print(f"  {nome:>12}: média {serie.mean():6.1f} | desvio padrão {serie.std():6.2f} "
          f"| amplitude {serie.max() - serie.min():5.0f}")

### Desvio padrão

O **desvio padrão** é a raiz da média dos desvios ao quadrado em relação à média. Em
palavras: *quanto, em média, os valores se afastam do centro*. A grande vantagem é estar
na **mesma unidade** dos dados — se os retornos estão em %, o desvio padrão está em %.

Em finanças ele tem nome próprio: **volatilidade**.

In [ ]:
resumo_risco = acoes.groupby("ticker")["retorno_diario"].agg(
    retorno_medio="mean",
    volatilidade="std",
).round(3)

# Anualizando: 252 pregões por ano
resumo_risco["retorno_anual_%"] = (resumo_risco["retorno_medio"] * 252).round(1)
resumo_risco["volatilidade_anual_%"] = (resumo_risco["volatilidade"] * np.sqrt(252)).round(1)

resumo_risco.sort_values("volatilidade_anual_%")

> **Atenção — Sobre a anualização:** multiplicar a média por 252 e o desvio por √252 são
> aproximações padrão de mercado, que supõem retornos independentes entre os dias. Elas
> servem para comparar ativos entre si, não como previsão. Aqui interessa a ordem do
> ranking, não a terceira casa decimal.

### IQR: a dispersão robusta

O **intervalo interquartil** (IQR) é a distância entre o 1º e o 3º quartil — a largura
da faixa que contém os 50% centrais. Ele ignora as caudas, e por isso não é afetado por
valores extremos. É a medida de dispersão que acompanha a mediana, assim como o desvio
padrão acompanha a média.

In [ ]:
q1 = patrimonio.quantile(0.25)
q3 = patrimonio.quantile(0.75)
iqr = q3 - q1

print(f"Q1  (25%) : R$ {q1:>12,.2f}")
print(f"Q3  (75%) : R$ {q3:>12,.2f}")
print(f"IQR       : R$ {iqr:>12,.2f}   <- faixa dos 50% centrais")
print(f"Desvio pad: R$ {patrimonio.std():>12,.2f}")

### Coeficiente de variação

Para comparar a dispersão de variáveis em escalas diferentes, o desvio padrão sozinho
não serve — R$ 5 de desvio significam coisas distintas em uma ação de R$ 10 e em outra
de R$ 500. O **coeficiente de variação** (desvio ÷ média) resolve isso: ele é
adimensional.

In [ ]:
precos = acoes.groupby("ticker")["fechamento_ajustado"].agg(["mean", "std"])
precos["coef_variacao"] = (precos["std"] / precos["mean"]).round(3)

precos.round(2).sort_values("coef_variacao", ascending=False)

## 3. Quantis, percentis e formato

Um **quantil** divide os dados ordenados em partes. O percentil 90 é o valor abaixo do
qual estão 90% das observações.

In [ ]:
for p in [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]:
    print(f"  percentil {p:>5.0%}: R$ {patrimonio.quantile(p):>12,.2f}")

> Em gestão de risco, o percentil 5 dos retornos tem nome: **VaR** (*Value at Risk*).
> "O VaR diário de 95% é −3,2%" significa: em 95% dos dias a perda não passa disso —
> e nos outros 5%, passa. Note o que a medida **não** diz: quanto se perde nesses 5%.

In [ ]:
retornos_petr = acoes.loc[acoes["ticker"] == "PETR4", "retorno_diario"].dropna()

var_95 = retornos_petr.quantile(0.05)
perda_media_na_cauda = retornos_petr[retornos_petr <= var_95].mean()

print(f"VaR 95% diário da PETR4      : {var_95:.2f}%")
print(f"Perda média nos piores 5% dias: {perda_media_na_cauda:.2f}%")
print(f"Pior dia do período          : {retornos_petr.min():.2f}%")

### Assimetria

A **assimetria** (*skewness*) mede o desequilíbrio entre as caudas:

- **positiva** → cauda longa à direita; média > mediana (patrimônio, renda);
- **≈ zero** → simétrica;
- **negativa** → cauda longa à esquerda; média < mediana.

In [ ]:
print(f"Assimetria do patrimônio       : {patrimonio.skew():.2f}")
print(f"Assimetria da idade            : {clientes['idade'].skew():.2f}")
print(f"Assimetria dos retornos (PETR4): {retornos_petr.skew():.2f}")

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 3.8))

sns.histplot(patrimonio, bins=35, ax=eixos[0], color="#c55a11")
eixos[0].set_title(f"Patrimônio — assimetria {patrimonio.skew():.2f}")
eixos[0].set_xlabel("R$")

sns.histplot(clientes["idade"], bins=25, ax=eixos[1], color="#1f4e79")
eixos[1].set_title(f"Idade — assimetria {clientes['idade'].skew():.2f}")
eixos[1].set_xlabel("anos")

sns.histplot(retornos_petr, bins=60, ax=eixos[2], color="#2e7d32")
eixos[2].set_title(f"Retornos PETR4 — assimetria {retornos_petr.skew():.2f}")
eixos[2].set_xlabel("%")

for ax in eixos:
    ax.set_ylabel("")
fig.tight_layout()
plt.show()

## 4. Outliers: extremo não é erro

Um **outlier** é uma observação muito distante das demais. O critério mais usado é a
regra do IQR: fica fora quem está abaixo de `Q1 − 1,5×IQR` ou acima de `Q3 + 1,5×IQR`.
É exatamente o que o boxplot desenha como pontos soltos.

In [ ]:
def detectar_outliers(serie):
    """Devolve uma máscara booleana marcando outliers pela regra do IQR."""
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)


outliers = detectar_outliers(patrimonio)

print(f"{outliers.sum()} outliers de {len(patrimonio)} ({outliers.mean():.1%})")
print(f"Menor outlier: R$ {patrimonio[outliers].min():,.2f}")
print(f"Eles concentram {patrimonio[outliers].sum() / patrimonio.sum():.1%} do patrimônio total")

Antes de "tratar" esses pontos, pergunte **o que eles são**:

| O ponto é… | O que fazer |
|---|---|
| **Erro de digitação ou de sistema** (idade 250, preço negativo) | corrigir ou transformar em `NaN` |
| **Valor real e extremo** (cliente com R$ 900 mil, crash de mercado) | **manter** — muitas vezes é o dado mais importante |
| **De outra população** (um fundo institucional na base de pessoas físicas) | separar a análise |

> **Atenção:** Remover outliers "para melhorar o gráfico" é uma das maneiras mais fáceis
> de produzir uma análise errada. Em finanças, os dias extremos **são** o objeto de
> estudo: foi exatamente neles que as carteiras quebraram. Uma análise de risco que
> descarta os piores dias não descreve risco nenhum.

In [ ]:
# Os cinco piores pregões do período — nenhum deles é erro
piores = acoes.dropna(subset=["retorno_diario"]).nsmallest(5, "retorno_diario")
piores[["data", "ticker", "fechamento_ajustado", "retorno_diario"]]

## 5. Correlação

A **correlação de Pearson** mede a força e a direção de uma relação **linear** entre
duas variáveis. Varia de −1 a +1:

- **+1** → quando uma sobe, a outra sobe, exatamente na mesma proporção;
- **0** → nenhuma relação linear;
- **−1** → quando uma sobe, a outra desce.

Uma referência aproximada para leitura: até 0,3 é fraca; 0,3 a 0,7 é moderada; acima de
0,7 é forte. Mas o contexto manda — em ciências sociais 0,4 já é muito; entre duas ações
do mesmo setor, 0,4 é pouco.

In [ ]:
print("Correlação idade × patrimônio:",
      round(clientes["idade"].corr(clientes["patrimonio_investido"]), 3))
print("Correlação patrimônio × aporte mensal:",
      round(clientes["patrimonio_investido"].corr(clientes["aporte_mensal"]), 3))

In [ ]:
# A matriz inteira, de uma vez
numericas = clientes[["idade", "patrimonio_investido", "aporte_mensal"]]
numericas.corr().round(3)

### Correlação só enxerga o que é linear

Este é o ponto mais importante da seção. Uma correlação próxima de zero **não** significa
"não há relação" — significa "não há relação **linear**".

In [ ]:
gerador = np.random.default_rng(42)
x = np.linspace(-3, 3, 300)

exemplos = {
    "Linear forte": (x, 2 * x + gerador.normal(0, 1, 300)),
    "Sem relação": (x, gerador.normal(0, 3, 300)),
    "Relação em U": (x, x ** 2 + gerador.normal(0, 1, 300)),
}

fig, eixos = plt.subplots(1, 3, figsize=(14, 4))

for ax, (titulo, (eixo_x, eixo_y)) in zip(eixos, exemplos.items()):
    correlacao = np.corrcoef(eixo_x, eixo_y)[0, 1]
    ax.scatter(eixo_x, eixo_y, alpha=0.5, s=14)
    ax.set_title(f"{titulo}\ncorrelação = {correlacao:.2f}")
    ax.set_xlabel("x")

eixos[0].set_ylabel("y")
fig.tight_layout()
plt.show()

No terceiro gráfico a relação é **perfeita e determinística** — y depende inteiramente de
x — e mesmo assim a correlação de Pearson fica perto de zero. Ela mede inclinação de
reta, e ali não há reta.

**Moral: sempre olhe o gráfico de dispersão antes de acreditar em um coeficiente.**

> A **correlação de Spearman** (`.corr(method="spearman")`) resolve parte disso: ela
> mede relação **monotônica** — se uma variável tende a crescer quando a outra cresce,
> mesmo que não em linha reta. É mais robusta a outliers e útil quando a relação é
> claramente curva mas sempre no mesmo sentido.

In [ ]:
print("Pearson :", round(clientes["patrimonio_investido"].corr(clientes["aporte_mensal"]), 3))
print("Spearman:", round(clientes["patrimonio_investido"].corr(clientes["aporte_mensal"],
                                                               method="spearman"), 3))

### Correlação não é causalidade

Se A e B andam juntos, há quatro explicações possíveis, e apenas uma é "A causa B":

1. **A causa B**;
2. **B causa A** (a direção pode ser a oposta da que você imaginou);
3. **Uma terceira variável C causa as duas** (o clássico "fator de confusão");
4. **Coincidência** — e, com bases grandes, coincidências são garantidas.

O caso 3 é o mais comum na prática. Exemplo canônico: vendas de sorvete e afogamentos
correlacionam-se fortemente. Sorvete não afoga ninguém — o verão causa os dois.

In [ ]:
# Um exemplo construído com os nossos dados, para ver a coincidência acontecer
retornos = acoes.pivot_table(index="data", columns="ticker", values="retorno_diario")

# Uma variável totalmente artificial, sem qualquer relação com o mercado
gerador = np.random.default_rng(7)
ruido = pd.Series(gerador.normal(0, 1, len(retornos)), index=retornos.index)

correlacoes_espurias = retornos.apply(lambda coluna: coluna.corr(ruido))
print("Correlação de uma série ALEATÓRIA com cada ativo:")
print(correlacoes_espurias.round(3))
print(f"\nMaior correlação em módulo: {correlacoes_espurias.abs().max():.3f}")

Correlações pequenas, como esperado — a série é ruído puro. Mas o exercício mostra o
mecanismo: se você testar **centenas** de variáveis contra a sua, alguma vai passar de
0,3 por acaso. É assim que nascem descobertas espúrias.

**A defesa não é estatística, é conceitual:** só investigue relações para as quais você
consegue formular um mecanismo plausível *antes* de olhar o número. Correlação é uma
pista, não uma conclusão.

## 6. Aplicação: o mapa risco × retorno

Juntando quase tudo da aula em um único gráfico — o mais tradicional da análise de
investimentos.

In [ ]:
mapa = acoes.groupby("ticker")["retorno_diario"].agg(["mean", "std"])
mapa["retorno_anual"] = mapa["mean"] * 252
mapa["risco_anual"] = mapa["std"] * np.sqrt(252)
mapa["retorno_por_risco"] = (mapa["retorno_anual"] / mapa["risco_anual"]).round(2)

fig, ax = plt.subplots(figsize=(9, 6.5))

ax.scatter(mapa["risco_anual"], mapa["retorno_anual"], s=140,
           color="#1f4e79", zorder=3)

for ticker, linha in mapa.iterrows():
    ax.annotate(ticker, (linha["risco_anual"], linha["retorno_anual"]),
                xytext=(9, 5), textcoords="offset points", fontsize=11)

ax.axhline(0, color="black", linewidth=0.9)
ax.set_title("Risco × retorno anualizados (2021–2025)", fontsize=13, pad=12)
ax.set_xlabel("Risco — volatilidade anualizada (%)")
ax.set_ylabel("Retorno anualizado médio (%)")
ax.grid(alpha=0.3)

plt.show()

In [ ]:
mapa[["retorno_anual", "risco_anual", "retorno_por_risco"]].round(1).sort_values(
    "retorno_por_risco", ascending=False
)

A leitura desse gráfico: quanto mais **acima** melhor (mais retorno), quanto mais à
**esquerda** melhor (menos risco). O canto superior esquerdo é o lugar cobiçado.

E a última coluna — retorno dividido por risco — é a ideia por trás do **índice de
Sharpe**: quanto de retorno cada unidade de risco entregou. É como se compara ativos
que não são comparáveis olhando só para o retorno.

> **Atenção:** Tudo isso é **descritivo**: descreve o que aconteceu entre 2021 e 2025. Não
> é previsão. Cinco anos são poucos, o período inclui condições muito particulares
> (pandemia, ciclo de juros, eleições), e retorno passado não se repete por decreto.
> Confundir descrição com previsão é o erro mais caro da área.

## 7. Recapitulando

- **Média** usa tudo mas é sensível a extremos; **mediana** é robusta. Em distribuições
  assimétricas, reporte as duas — a distância entre elas é informação.
- **Desvio padrão** acompanha a média; **IQR** acompanha a mediana. **Coeficiente de
  variação** permite comparar escalas diferentes.
- **Quantis** descrevem a distribuição inteira. O percentil 5 dos retornos é o VaR.
- **Assimetria** mede o desequilíbrio das caudas.
- **Outlier ≠ erro.** Investigue antes de remover; em risco, os extremos são o objeto.
- **Correlação** mede relação **linear** entre −1 e +1. Sempre confira com um gráfico de
  dispersão. Spearman capta relações monotônicas não lineares.
- **Correlação não é causalidade** — pense no mecanismo antes de olhar o número.
- Estatística descritiva descreve o passado. Não é previsão.

**Próxima aula:** juntar tudo isso em um processo — a Análise Exploratória de Dados.